### Preprocessing

In [1]:
import numpy as np
from sklearn.utils import shuffle
from skimage import exposure

In [2]:
def preprocessing_dataset(X, y = None):
    X = 0.299 * X[:, :, :, 0] + 0.587 * X[:, :, :, 1] + 0.114 * X[:, :, :, 2]
    X = (X / 255.).astype(np.float32)
    for i in range(X.shape[0]):
        X[i] = exposure.equalize_adapthist(X[i])
    if y is not None:
        y = np.eye(43)[y]
        X, y = shuffle(X, y)
    X = X.reshape(X.shape + (1,))
    return X, y

### Augmentation

In [3]:
from skimage.transform import rotate
from skimage.transform import warp
from skimage.transform import ProjectiveTransform

In [4]:
def flip_extend(X, y):
    horizontally_flippable = np.array([11, 12, 13, 15, 17, 18, 22, 26, 30, 35])
    vertically_flippable = np.array([1, 5, 12, 15, 17])
    both_flippable = np.array([32, 40])
    cross_flippable = np.array([
        [19, 20], 
        [33, 34], 
        [36, 37], 
        [38, 39],
        [20, 19], 
        [34, 33], 
        [37, 36], 
        [39, 38],
    ])
    n_classes = 43
    X_extend = np.empty([0, X.shape[1], X.shape[2], X.shape[3]], dtype = X.dtype)
    y_extend = np.empty([0], dtype = y.dtype)
    for c in range(n_classes):
        X_extend = np.append(X_extend, X[y==c], axis=0)
        if c in horizontally_flippable:
            X_extend = np.append(X_extend, X[y==c][:, :, ::-1, :], axis=0)
        if c in cross_flippable[:, 0]:
            flip_class = cross_flippable[cross_flippable[:, 0] == c][0][1]
            X_extend = np.append(X_extend, X[y==flip_class][:, :, ::-1, :], axis=0)
        y_extend = np.append(y_extend, np.full((X_extend.shape[0] - y_extend.shape[0]), c, dtype=int))
        if c in vertically_flippable:
            X_extend = np.append(X_extend, X_extend[y_extend == c][:, ::-1, :, :], axis=0)
        y_extend = np.append(y_extend, np.full((X_extend.shape[0] - y_extend.shape[0]), c, dtype=int))
        if c in both_flippable:
            X_extend = np.append(X_extend, X_extend[y_extend == c][:, ::-1, ::-1, :], axis=0)
        y_extend = np.append(y_extend, np.full((X_extend.shape[0] - y_extend.shape[0]), c, dtype=int))
    return (X_extend, y_extend)

In [ ]:
def apply_rotate(X, intensity):
    for i in range(X.shape[0]):
        delta = 30 * intensity
        X[i] = rotate(X[i], np.random.uniform(-delta, delta), mode="edge")
    return X

In [1]:
def apply_projection(X, intensity):
    image_size = X.shape[1]
    delta = image_size * 0.3 * intensity
    for i in range(X.shape[0]):
        tl_top = np.random.uniform(-d, d)
        tl_left = np.random.uniform(-d, d)
        bl_bottom = np.random.uniform(-d, d)
        bl_left = np.random.uniform(-d, d)
        tr_top = np.random.uniform(-d, d)
        tr_right = np.random.uniform(-d, d)
        br_bottom = np.random.uniform(-d, d)
        br_right = np.random.uniform(-d, d)
        transform = ProjectiveTransform()
        transform.from_estimate(
            np.array((
                (tl_left, tl_top),
                (bl_left, image_size - bl_bottom),
                (image_size - br_right, image_size - br_bottom),
                (image_size - tr_right, tr_top)
            )),
            np.array((
                (0, 0),
                (0, image_size),
                (image_size, image_size),
                (image_size, 0)
            )))
        X[i] = warp(X[1], transform, output_shape=(image_size, image_size), order=1, mode="edge")
    return X
